# Belgrade Air Quality - Data Exploration

First look at hourly PM2.5 and PM10 data from 5 reference stations in Belgrade
(source: OpenAQ API, data provided by EEA). The goal is to understand the data
and find quality issues before cleaning.

## Setup

In [1]:
import pandas as pd

## Load data
Raw data downloaded with `src/fetch_data.py`.

In [2]:
df = pd.read_csv("../data/raw/belgrade_air_quality_raw.csv", parse_dates=["datetime_utc"])


## Basic overview

In [14]:
df.head()

,datetime_utc,value,station,location_id,parameter
0,2021-01-26 23:00:00+00:00,34.5,Novi Beograd,10879,pm25
1,2021-01-27 00:00:00+00:00,36.1,Novi Beograd,10879,pm25
2,2021-01-27 01:00:00+00:00,31.5,Novi Beograd,10879,pm25
3,2021-01-27 02:00:00+00:00,29.9,Novi Beograd,10879,pm25
4,2021-01-27 03:00:00+00:00,30.7,Novi Beograd,10879,pm25


In [4]:
df.shape

(360765, 5)

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 360765 entries, 0 to 360764
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype              
---  ------        --------------   -----              
 0   datetime_utc  360765 non-null  datetime64[us, UTC]
 1   value         360765 non-null  float64            
 2   station       360765 non-null  str                
 3   location_id   360765 non-null  int64              
 4   parameter     360765 non-null  str                
dtypes: datetime64[us, UTC](1), float64(1), int64(1), str(2)
memory usage: 13.8 MB


- 360,765 rows, 5 columns
- No missing values
- `datetime_utc` is in UTC time and needs to be converted to Belgrade local time


## Value distribution


In [6]:
df.describe()

,value,location_id
count,360765.000000,360765.000000
mean,26.351648,96889.511294
std,23.926694,242814.847857
min,0.000000,10837.000000
25%,11.600000,10837.000000
50%,19.700000,10879.000000
75%,32.700000,11587.000000
max,821.000000,784164.000000


Minimum value is 0 and maximum is 821 µg/m³. Both need a closer look:
zero is unrealistic for city air, and 821 is extremely high.

## Rows per station and pollutant


In [7]:
df.groupby(["station", "parameter"]).size()

station           parameter
Mostar            pm10         40826
                  pm25         39323
Novi Beograd      pm10         59509
                  pm25         41268
Obrenovac Centar  pm10         20013
                  pm25         20021
Stari grad        pm10         59941
                  pm25         51879
Vracar            pm10         13994
                  pm25         13991
dtype: int64

Stations have very different amounts of data. Vračar has only ~14,000 rows
although it has been reporting since 2020, so a lot of its data is missing.

## Duplicates
Checked both full-row duplicates and duplicates on station + pollutant + hour.

In [8]:
df.duplicated().sum()

np.int64(0)

In [9]:
df.duplicated(subset=["station", "parameter", "datetime_utc"]).sum()

np.int64(0)

Result: no duplicates.

## Zero values


In [15]:
df[df["value"] == 0].groupby("station").size()   


station
Mostar          312
Novi Beograd    142
Stari grad       22
dtype: int64

476 hours with value 0 (Mostar 312, Novi Beograd 142, Stari grad 22).
These are most likely sensor errors and will be removed.

## Extreme values
Looking at the 10 highest values and checking whether they are sensor errors
or real pollution events.

In [11]:
df.sort_values("value", ascending=False).head(10) # 10 najvećih vrednosti

,datetime_utc,value,station,location_id,parameter
67046,2022-03-16 05:00:00+00:00,821.0,Novi Beograd,10879,pm10
159381,2018-10-18 05:00:00+00:00,676.0,Stari grad,10837,pm10
321647,2023-05-31 18:00:00+00:00,656.0,Obrenovac Centar,784164,pm10
230965,2023-07-19 18:00:00+00:00,561.0,Mostar,11587,pm10
170403,2020-03-27 06:00:00+00:00,555.0,Stari grad,10837,pm10
170399,2020-03-27 02:00:00+00:00,528.0,Stari grad,10837,pm10
170404,2020-03-27 07:00:00+00:00,521.0,Stari grad,10837,pm10
170401,2020-03-27 04:00:00+00:00,518.0,Stari grad,10837,pm10
170402,2020-03-27 05:00:00+00:00,516.0,Stari grad,10837,pm10
170405,2020-03-27 08:00:00+00:00,489.0,Stari grad,10837,pm10


**Findings:**
- **Isolated spikes** - one very high hour between normal values
  (e.g. Obrenovac, 656 µg/m³ on 2023-05-31) → most likely a sensor error, remove.
- **Multi-hour events** - high values for several hours in a row → real events, keep.

Two real events confirmed by news reports:
- **27 March 2020, Stari grad (up to 555 µg/m³ PM10)** - desert dust from the
  Karakum desert ([021.rs](https://www.021.rs/info/srbija/238036/foto-pesak-iz-pustinje-karakum-podigao-nivo-zagadjenja-vazduha-u-srbiji)).
- **16 March 2022, Novi Beograd (821 µg/m³ PM10)** - Saharan dust; the same peak
  value was reported in the news ([Balkan Green Energy News](https://balkangreenenergynews.com/rs/pesak-iz-sahare-doneo-rekordno-zagadjenje-vazduha-ugrozene-ranjive-grupe/)).

### Checking the Obrenovac spike


In [12]:
mask = (df["station"] == "Obrenovac Centar") & (df["datetime_utc"].between("2023-05-31T12:00", "2023-06-01T00:00"))
df[mask]

,datetime_utc,value,station,location_id,parameter
321642,2023-05-31 12:00:00+00:00,12.30,Obrenovac Centar,784164,pm10
321643,2023-05-31 13:00:00+00:00,13.10,Obrenovac Centar,784164,pm10
321644,2023-05-31 14:00:00+00:00,11.30,Obrenovac Centar,784164,pm10
321645,2023-05-31 16:00:00+00:00,19.70,Obrenovac Centar,784164,pm10
321646,2023-05-31 17:00:00+00:00,16.20,Obrenovac Centar,784164,pm10
321647,2023-05-31 18:00:00+00:00,656.00,Obrenovac Centar,784164,pm10
321648,2023-05-31 19:00:00+00:00,9.18,Obrenovac Centar,784164,pm10
321649,2023-05-31 21:00:00+00:00,9.89,Obrenovac Centar,784164,pm10
321650,2023-05-31 22:00:00+00:00,11.70,Obrenovac Centar,784164,pm10
321651,2023-05-31 23:00:00+00:00,13.40,Obrenovac Centar,784164,pm10


The hour before and after the spike show normal values (9–20 µg/m³), which confirms it is an isolated sensor error.

## PM2.5 vs PM10 consistency
PM2.5 particles are a subset of PM10, so PM2.5 cannot be higher than PM10
at the same station and hour. 

In [13]:
wide = df.pivot_table(index=["station", "datetime_utc"], columns="parameter", values="value")
(wide["pm25"] > wide["pm10"]).sum()

np.int64(585)

Found 585 such hours, which are measurement errors.

## Data period per station
Checking the first and last available hour for each station and pollutant.

In [17]:
df.groupby(["station", "parameter"])["datetime_utc"].agg(["min", "max"])

min                       max
station          parameter                                                    
Mostar           pm10      2020-11-06 22:00:00+00:00 2026-11-18 13:00:00+00:00
                 pm25      2021-01-26 23:00:00+00:00 2026-11-18 13:00:00+00:00
Novi Beograd     pm10      2017-10-02 10:00:00+00:00 2026-11-18 12:00:00+00:00
                 pm25      2021-01-26 23:00:00+00:00 2026-11-18 12:00:00+00:00
Obrenovac Centar pm10      2023-03-30 17:00:00+00:00 2026-11-18 13:00:00+00:00
                 pm25      2023-03-30 17:00:00+00:00 2026-11-18 13:00:00+00:00
Stari grad       pm10      2017-09-22 08:00:00+00:00 2026-11-18 12:00:00+00:00
                 pm25      2019-03-17 12:00:00+00:00 2026-11-18 12:00:00+00:00
Vracar           pm10      2023-03-30 17:00:00+00:00 2025-12-02 18:00:00+00:00
                 pm25      2023-03-30 17:00:00+00:00 2025-12-02 18:00:00+00:00

**Findings:**
- Stations have different data periods, so each question uses its own period.
- Longest records: Novi Beograd and Stari grad (PM10 from 2017).
- PM2.5 starts later: Stari grad 2019, Novi Beograd and Mostar 2021.
- Vračar reports only from 2023-03-30 to 2025-12-02 (no recent data).
- Common period for all 5 stations: 2023-03-30 to 2025-12-02.

## Data coverage per year
Coverage = hours with data / 8,760 hours in a year × 100.
Years with coverage below 75% are not reliable for comparison.

In [19]:
df["year"] = df["datetime_utc"].dt.year 
counts = df.groupby(["station", "parameter", "year"]).size()
coverage = (counts / 8760 * 100).round(1)
coverage.unstack("year").fillna(0).T

station   Mostar       Novi Beograd       Obrenovac Centar       Stari grad  \
parameter   pm10  pm25         pm10  pm25             pm10  pm25       pm10   
year                                                                          
2017         0.0   0.0          0.0   0.0              0.0   0.0       14.1   
2018         0.0   0.0         48.2   0.0              0.0   0.0       77.7   
2019         0.0   0.0         75.7   0.0              0.0   0.0       88.2   
2020        12.2   0.0         79.0   0.0              0.0   0.0       61.3   
2021        67.9  62.6         73.5  68.2              0.0   0.0       63.7   
2022        90.7  90.9         90.1  90.1              0.0   0.0       89.4   
2023        82.5  82.5         82.4  82.4             29.7  29.7       82.7   
2024        85.3  85.2         87.0  87.0             79.6  79.7       88.3   
2025        56.8  56.8         72.3  72.3             59.6  59.6       52.8   
2026        70.7  71.0         71.2  71.2             59.5  59.5       65.9   

station         Vracar        
parameter  pm25   pm10  pm25  
year                          
2017        0.0    0.0   0.0  
2018        0.0    0.0   0.0  
2019       71.7    0.0   0.0  
2020       77.6    0.0   0.0  
2021       63.7    0.0   0.0  
2022       89.5    0.0   0.0  
2023       82.7   29.9  29.9  
2024       88.3   70.3  70.2  
2025       52.8   59.6  59.6  
2026       65.9    0.0   0.0

**Findings:**
- 2024 is the only year with data from all 5 stations (70–88% coverage).
  Vračar is slightly below 75% (70%).
- 2025 has low coverage at all stations (53–72%) - needs a monthly check.
- Long-term records have gaps (e.g. Stari grad 2020–2021 below 75%),
  so trends will be based on monthly averages instead of yearly ones.

**Decision:**
- Station comparison: year 2024.
- Trend: monthly averages for Novi Beograd and Stari grad, using only months
  with at least 75% of hours.

## Monthly coverage in 2025
2025 has low yearly coverage at all stations, so I checked which months are missing.

In [23]:
df['month'] = df['datetime_utc'].dt.month
monthly = df.groupby(["station", "parameter", "year", "month"]).size()

monthly

station  parameter  year  month
Mostar   pm10       2020  11       529
                          12       544
                    2021  1        574
                          2        453
                          3        637
                                  ... 
Vracar   pm25       2025  8        158
                          9        714
                          10       229
                          11       295
                          12        37
Length: 653, dtype: int64

In [25]:
mask = (df["station"] == "Novi Beograd") & (df["parameter"] == "pm10") & (df["year"] == 2025)
df[mask].groupby("month").size()

month
1     570
2     617
3     694
4     496
5     522
6     623
8     133
9     614
10    693
11    697
12    676
dtype: int64

In [26]:
mask_2025 = (df["year"] == 2025) & (df["parameter"] == "pm10")
df[mask_2025].groupby(["month", "station"]).size().unstack("station")

station,Mostar,Novi Beograd,Obrenovac Centar,Stari grad,Vracar
month,,,,,
1,419,570,602,541,603
2,600,617,669,610,670
3,725,694,725,653,725
4,532,496,532,487,531
5,561,522,555,506,555
6,703,623,703,632,703
8,158,133,158,140,158
9,714,614,714,599,714
10,229,693,230,183,229


In [27]:
import os, requests
from dotenv import load_dotenv

load_dotenv("../.env")
headers = {"X-API-Key": os.getenv("OPENAQ_API_KEY")}

url = "https://api.openaq.org/v3/sensors/36424/hours"   # Novi Beograd, PM10
params = {"datetime_from": "2025-07-01T00:00:00Z",
          "datetime_to": "2025-08-01T00:00:00Z",
          "limit": 1000}

response = requests.get(url, headers=headers, params=params)
print(response.status_code, len(response.json()["results"]))

200 0


**Findings:**
- July 2025 has no data at any station, and August 2025 has only 133–158 hours.
- October and November 2025 are also low at all stations except Novi Beograd.
- Mostar, Obrenovac and Vračar have almost identical gaps, which points to a
  problem at the data source, not at individual stations.
- A direct API request for July 2025 returned status 200 with 0 results,
  confirming the data is missing at the source (not a download error).

**Decision:**
- 2025 is excluded from year-to-year comparisons.
- Monthly trends use only months with at least 75% of hours available.

## Negative values
Air pollution cannot be negative, so any negative value would be a sensor error.

In [29]:
(df["value"] < 0).sum()

np.int64(0)

**Findings:** no negative values found.

## Summary

The raw dataset has 360,765 hourly measurements of PM2.5 and PM10 from
5 reference stations in Belgrade (2017–2026). Key findings:

- **No missing values or duplicates** in the downloaded data.
- **Stations have different data periods.** Novi Beograd and Stari grad have
  the longest PM10 records (from 2017). Obrenovac and Vračar start in 2023,
  and Vračar stops in December 2025.
- **Data coverage varies by year.** 2024 is the only year with good coverage
  at all 5 stations. 2025 has gaps at the data source (no data for July,
  very little for August).
- **Data quality issues:** 476 zero values, 585 hours where PM2.5 > PM10,
  and isolated spikes that are sensor errors.
- **Real pollution events:** extreme values on 27 March 2020 (Karakum desert
  dust) and 16 March 2022 (Saharan dust, 821 µg/m³ PM10 in Novi Beograd)
  are real events confirmed by news reports and will be kept.

## Cleaning plan
1. Convert `datetime_utc` to Belgrade local time
2. Remove zero values
3. Remove PM2.5 values where PM2.5 > PM10
4. Remove isolated spikes, keep multi-hour events
5. Flag months with less than 75% of hours, so they can be excluded from trends

## Analysis decisions
- **Station comparison:** year 2024 (all 5 stations).
- **Trends:** monthly averages for Novi Beograd and Stari grad, using only
  months with at least 75% of hours.
- **2025** is excluded from year-to-year comparisons.